# 07 — Driving a real town

Everything up to here happened on a graph, where a leg takes exactly as long
as the model says. This notebook puts the same policy in a vehicle in
**CARLA**, where a leg takes as long as the driving actually takes.

**This notebook needs a running CARLA server.** Cells that need one check
first and skip themselves with a message if it is not there, so the notebook
can still be read end to end without it.

```bash
docker compose -f compose_software.yml up -d      # start the server
```

Driving is not what this project is about — the design review lists vehicle
control as a *risk*, mitigated by "CARLA's built-in autonomous driving should
be sufficient for a PoC", and scopes it inside the backend. What follows is
therefore a tour of that layer, not a contribution: **we did not write a
self-driving car, we wrote the layer that tells an existing one where to go.**

Runtime: a few minutes with a server, seconds without.

In [ ]:
# --- Bootstrap ------------------------------------------------------------
# This notebook imports the project as `src.…`, which requires the repository
# root on the import path. `./jupyter-run.sh` puts it there; opening the file
# any other way does not. This cell finds the root by walking up from the
# working directory, so the notebook runs wherever it is launched from and
# whatever folder it sits in.
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Seeded runs only reproduce when Python's string hashing is pinned: several
# policies break ties with min() over a set, whose iteration order is
# randomised per process otherwise. The simulation refuses to run unpinned;
# this says so now rather than after a long cell.
if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print("WARNING: PYTHONHASHSEED is not pinned, so seeded runs will be "
          "refused.\n         Launch with ./jupyter-run.sh, or set "
          "PYTHONHASHSEED=0.")

print(f"repository root: {ROOT}")

In [ ]:
import os

# How much work this notebook does. Every notebook in this folder has this
# switch and honours the AUTOBUS_SCALE environment variable, so nothing here
# needs editing to run it bigger or smaller.
#
#   "quick"  the default: minutes, same conclusions, wider error bars
#   "full"   the sample sizes behind the figures in the submitted documents
#   "smoke"  a tiny end-to-end check that the notebook runs at all
#
# The outputs saved in this file were produced at "full".
SCALE = os.environ.get("AUTOBUS_SCALE", "quick")
print(f"running at SCALE = {SCALE!r}")

In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

In [ ]:
import socket

CARLA_HOST = os.environ.get("CARLA_HOST", "localhost")
CARLA_PORT = int(os.environ.get("CARLA_PORT", "2000"))


def server_is_up(host=CARLA_HOST, port=CARLA_PORT, timeout=2.0):
    """
    Report whether something is listening on the CARLA port.
    :param host: CARLA host.
    :param port: CARLA RPC port.
    :param timeout: Seconds to wait for the connection.
    :return: True when the port accepts a connection.
    """
    try:
        with socket.create_connection((host, port), timeout):
            return True
    except OSError:
        return False


SERVER = server_is_up()
print(f"CARLA at {CARLA_HOST}:{CARLA_PORT} -- "
      + ("up" if SERVER else "NOT REACHABLE"))
if not SERVER:
    print("\nCells that need the server will skip themselves and say so.\n"
          "Start it with:  docker compose -f compose_software.yml up -d")

## 1. What changes, and what does not

Exactly one thing changes: the **backend**.

| Layer | Graph world | CARLA world |
|---|---|---|
| Policy | identical | identical |
| Bus agent, request pool, demand, metrics | identical | identical |
| Backend | `MockBackend` — advances by dead reckoning | `CarlaBackend` — drives a vehicle |

`SimulationBackend` is three methods: take a destination, advance time, say
where the bus is. Everything above that seam is untouched, which is what makes
comparing the two worlds meaningful rather than comparing two programs.

In [ ]:
from src.simulation.SimulationBackend import SimulationBackend

print("A backend must implement:")
for name in sorted(SimulationBackend.__abstractmethods__):
    print(f"  {name}")

## 2. The town is already prepared

Two artefacts per town live in the repository, so routing and analysis need no
server at all:

- **the topology dump** — every road, its length and its speed limit, read off
  a running server once by `src/preparation/carla_dump.py`;
- **the authored stop list** — where the bus stops, which is a human decision.

Plus one measured number, `traffic_efficiency`, discussed in section 5.

In [ ]:
from src.map.CarlaMapProvider import CarlaMapProvider

TOWN = "TAU"          # try "Town01", "Town03", "Town05", or "TAU"

provider = CarlaMapProvider(TOWN)
print(f"{TOWN}: {len(provider.get_stops())} stops, "
      f"{len(provider.get_road_edges())} road segments, "
      f"traffic_efficiency = {provider.traffic_efficiency():.4f}")
problems = provider.validate()
print(f"map problems: {len(problems)}")
for problem in problems[:3]:
    print(f"  {problem}")

## 3. How the bus gets from one stop to the next

CARLA has no "drive to this place" API, so this is the part the project had to
build. Three steps:

```
plan_route(map, origin, destination)  ->  waypoints along the lanes
agent.set_global_plan(waypoints)      ->  CARLA's own agent takes the route
bus.apply_control(agent.run_step())   ->  every tick, 20 per simulated second
```

`plan_route` is a Dijkstra search over CARLA's lane graph: from the current
waypoint, repeatedly ask for the waypoints 2 m further along, which follows
lane direction and expands every branch at a junction — so one-way roads and
legal turns come for free. It minimises **travel time**, not distance, because
the policy prices legs in time and a distance-optimal drive would be something
the policy never costed.

The single most important property of this design: **the planned waypoints
*are* the trajectory.** Nothing downstream re-derives, re-interprets or
second-guesses the route. That sounds unremarkable and is the conclusion of a
month of failures in which something did.

The driving itself is CARLA's `BasicAgent`: a PID controller for throttle,
brake and steering that follows a given route, stops at red lights and brakes
for vehicles ahead.

In [ ]:
if not SERVER:
    print("skipped: needs a live CARLA server")
else:
    import carla

    from src.simulation.carla_routing import plan_route
    from src.simulation.CarlaBackend import speed_limits_from, stop_positions_from
    from src.simulation import carla_world

    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(120.0)
    carla_world.load(client, TOWN)
    carla_map = client.get_world().get_map()

    positions = stop_positions_from(provider)
    stop_ids = list(positions)[:2]
    origin, destination = (carla.Location(*positions[s]) for s in stop_ids)

    route = plan_route(carla_map, origin, destination,
                       speed_for=speed_limits_from(provider),
                       lane_changes=True)
    print(f"{stop_ids[0]}  ->  {stop_ids[1]}")
    print(f"{len(route)} waypoints, "
          f"{2.0 * len(route):.0f} m of road at 2 m per step")

In [ ]:
if not SERVER:
    print("skipped: needs a live CARLA server")
else:
    xs = [w.transform.location.x for w in route]
    ys = [w.transform.location.y for w in route]
    figure, axis = plt.subplots(figsize=(6.4, 5.6))
    axis.plot(xs, ys, lw=2.2, color=SERIES[0])
    axis.scatter([xs[0]], [ys[0]], s=80, color=SERIES[2], zorder=3,
                 label="origin")
    axis.scatter([xs[-1]], [ys[-1]], s=80, color=SERIES[1], zorder=3,
                 label="destination")
    axis.set_title(f"One planned leg in {TOWN}")
    axis.set_aspect("equal")
    axis.legend()
    axis.set_xticks([])
    axis.set_yticks([])
    figure.tight_layout()

## 4. A whole trial, driven

Same `run_trial` as every other notebook. The only difference is a
`backend_factory` that builds a `CarlaBackend` instead of the mock.

Kept small on purpose: a CARLA trial runs in something close to real time, so
a handful of requests is minutes of driving.

In [ ]:
REQUESTS = {"smoke": 3, "quick": 6, "full": 10}[SCALE]

if not SERVER:
    print("skipped: needs a live CARLA server")
else:
    from tqdm.auto import tqdm

    from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
    from src.experiment import run_trial

    def build_backend(graph, start_stop_id):
        """Build the CARLA world for this trial."""
        from src.simulation.CarlaBackend import CarlaBackend
        return CarlaBackend(
            stop_positions=positions, position=start_stop_id, town=TOWN,
            # Must match how TransitGraph prices legs, or the policy and the
            # bus disagree about what a leg costs.
            speed_for=speed_limits_from(provider), lane_changes=True,
            host=CARLA_HOST, port=CARLA_PORT,
            fixed_delta_s=0.05,
            dwell_s=5.0,            # seconds stopped while passengers board
            npc_vehicles=20,        # traffic to queue behind
            seed=0, no_rendering=True, reload_world=True)

    # A CARLA trial runs close to real time, so this is minutes of driving
    # with nothing to show for it until the end. The bar advances as
    # passengers are delivered or give up; the observer is excluded from a
    # run's identity, so watching cannot change what is being watched.
    bar = tqdm(total=REQUESTS, desc=f"driving {TOWN}", unit="req")
    seen = {"settled": 0}

    def observe(progress):
        """Move the bar whenever another passenger reaches a final state."""
        if progress.settled == seen["settled"]:
            return
        seen["settled"] = progress.settled
        bar.n = progress.settled
        # refresh=False, then one refresh: set_postfix_str redraws by
        # default, which would render every update twice.
        bar.set_postfix_str(f"{progress.sim_time / 60:.0f} min simulated, "
                            f"{progress.pending} waiting", refresh=False)
        bar.refresh()

    carla_trial = run_trial(
        on_tick=observe,
        policy_factory=lambda g: ReorderingInsertionPolicy(g, max_wait_s=750.0),
        provider=provider,
        backend_factory=build_backend,
        seed=0, max_requests=REQUESTS, demand_interval_s=120.0,
        capacity=20, max_wait_s=750.0, dwell_s=5.0,
        dt=1.0, max_time=15_000.0, silent=True,
    )
    bar.close()
    print(f"served {carla_trial.kpi.served_fraction:.0%}  "
          f"wait {carla_trial.kpi.mean_wait_time:.0f} s  "
          f"ride {carla_trial.kpi.mean_ride_time:.0f} s")

### What the backend reports about itself

A CARLA trial can succeed on paper while the bus was in trouble the whole
time. `backend_stats` is how a healthy run is told apart from a salvaged one —
distance actually driven, how often the route had to be re-planned because the
bus was stuck, how many times it had to be nudged free.

**A campaign that does not look at these is not measuring routing.**

In [ ]:
if not SERVER:
    print("skipped: needs a live CARLA server")
else:
    for name, value in sorted(carla_trial.backend_stats.items()):
        print(f"  {name:<28} {value:g}")

## 5. Why CARLA needs a fudge factor

The policy plans against free-flow travel times from the graph. A real bus is
much slower than that: it waits at lights, queues behind traffic and
decelerates into junctions.

`traffic_efficiency` is one number per town that reconciles the two, and it is
**fitted by driving** — three full tours of every stop pair, with traffic. The
values range from about **0.35 to 0.64**, so the bus achieves between a third
and two thirds of the free-flow estimate, with the spread driven almost
entirely by how much of the time is spent at red lights.

Two consequences:

- Because it is fitted by driving, it belongs to *how* the bus drives. Change
  the steering and every town must be re-calibrated — which is why the stored
  run identity carries a steering tag.
- It multiplies every route equally, so it deliberately plays no part in
  choosing which route is quickest.

In [ ]:
import pandas as pd

display(pd.DataFrame([
    {"town": town,
     "traffic_efficiency": round(CarlaMapProvider(town).traffic_efficiency(), 4)}
    for town in ["Town01", "Town02", "Town03", "Town05", "TAU"]
]).set_index("town"))

print("Re-fit a town with:")
print("  ./run.sh -m src.preparation.carla_calibrate --town Town01 --tours 3")

## 6. Watching it drive

For a demonstration rather than a measurement, the project has an entry point
that opens a chase-camera window, draws the planned route on the road and
prints a summary when the bus is done:

```bash
./run.sh main.py                          # TAU, the recommended policy, a window
./run.sh main.py --policy fixed           # the baseline, for contrast
./run.sh main.py --help                   # every knob
```

It calls `run_trial` directly and records nothing, so it can never collide
with a campaign's stored results.

The same visual machinery is available from a notebook via `CarlaBackend`'s
`viewer`, `follow_camera` and `debug_draw` options — this is what produced the
screenshots in the submitted document.

## 7. What this design gave up

The bus is no longer driven by CARLA's traffic manager, so **it cannot change
lanes and cannot overtake.** It queues behind slow traffic instead of going
around it.

That is a real cost, and it is stated rather than hidden: with 20 NPC vehicles,
the within-leg spread of a leg's duration rose from 0.028 to between 0.04 and
0.19 — a leg now takes however long the traffic it happened to meet allows. It
is **policy-neutral**, because every arm of a comparison meets the same traffic
under the same seed, but it widens confidence intervals, and the success
criteria constrain those.

It was still the right trade. Over 112 legs across four towns:

| scheme | legs arrived | simulated time | distance driven |
|---|---|---|---|
| the earlier turn-list scheme | 53/56 | 16,440 s | 74,808 m |
| **driving the planned waypoints** | **56/56** | **7,434 s** | **40,035 m** |

Twice the reliability at half the driving.

Next: **08 — the CARLA campaign**, which asks whether the routing result from
notebook 05 survives all of this.